# Xenomorfo-Q: demostración de la directora

**SI3003 Inteligencia Artificial, Universidad EAFIT, 2026-2.**

## Problema

En un juego de sigilo, la "directora" conoce la posición real del jugador y decide cuándo ayudar a la criatura que lo persigue. Revisar una partida a mano (dónde está cada uno, cuántos pasos los separan, si la criatura ya vio al jugador) es lento. Este prototipo es un agente que responde esas preguntas en español sobre una partida simulada en la estación Vela-9.

## Opción B: agente con herramientas

Un modelo de lenguaje servido por NVIDIA NIM (`meta/llama-3.3-70b-instruct`) decide qué herramientas pedir. Las herramientas son funciones de Python sobre nuestra simulación:

| Herramienta | Qué hace |
|---|---|
| `estado_partida` | Tick, posición y sala del tripulante y del alienígena, pistas restantes y si hubo captura. |
| `ruta_mas_corta` | Pasos y salas del camino más corto (A*) entre dos salas o personajes. |
| `predecir_destino` | Sala hacia la que parece ir el tripulante, con su confianza. |
| `ve_tripulante` | Si el alienígena ve al tripulante y a cuántos pasos está. |
| `dar_pista` | Indica al alienígena una sala donde buscar (máximo 3 por partida). |
| `avanzar_simulacion` | Avanza de 1 a 20 ticks y devuelve los eventos. |

**Límite de control:** como máximo 6 herramientas ejecutadas por pregunta. Al llegar al límite, el modelo responde sin herramientas con lo que ya obtuvo y dice que alcanzó el límite.

## Preparación

Funciona en Colab y en local. En Colab clona el repositorio, instala las dependencias y lee la clave del secreto `NVIDIA_API_KEY` (icono de llave en la barra lateral). En local lee la clave del archivo `.env` de la raíz del repositorio. La clave nunca se muestra: solo se indica si está configurada.

In [ ]:
import os
import sys
import subprocess

EN_COLAB = "google.colab" in sys.modules

if EN_COLAB:
    if not os.path.exists("Xenomorfo-Q"):
        subprocess.run(["git", "clone", "https://github.com/jalvarez01/Xenomorfo-Q"], check=True)
    os.chdir("Xenomorfo-Q")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    # En local el notebook está en notebooks/; los paquetes están en la raíz.
    os.chdir("..")
sys.path.insert(0, os.getcwd())

if EN_COLAB:
    from google.colab import userdata
    try:
        os.environ["NVIDIA_API_KEY"] = userdata.get("NVIDIA_API_KEY") or ""
    except Exception:
        pass  # sin el secreto, la directora explica cómo configurarlo
else:
    from dotenv import load_dotenv
    # Ruta explícita: la búsqueda automática de .env depende de cómo se ejecuta la celda.
    load_dotenv(os.path.join(os.getcwd(), ".env"))

print("clave configurada:", "sí" if os.environ.get("NVIDIA_API_KEY", "").strip() else "no")

## Las herramientas sin el modelo

Las herramientas se pueden llamar directamente sobre una partida. Esto no usa la clave ni el modelo, y muestra lo que el modelo recibe cuando pide cada una. La partida usa la semilla 42.

In [ ]:
import json

from director import Partida, crear_herramientas

def mostrar(valor):
    print(json.dumps(valor, ensure_ascii=False, indent=2))

partida = Partida(semilla=42)
herramientas = crear_herramientas(partida)
mostrar(herramientas["estado_partida"]())
mostrar(herramientas["ruta_mas_corta"]("alienigena", "tripulante"))
mostrar(herramientas["avanzar_simulacion"](10))
mostrar(herramientas["estado_partida"]())

## Preguntas a la directora

Cada pregunta usa una directora nueva sobre una partida nueva con la semilla 42, igual que el ejecutor de casos. Debajo de cada respuesta aparece el registro: cada herramienta que pidió el modelo, con sus argumentos, su resultado y el tiempo, el número de llamadas y si se alcanzó el límite.

In [ ]:
from director import Directora, formatear_registro

def preguntar(pregunta, semilla=42, limite=6):
    directora = Directora(semilla=semilla, limite=limite)
    resultado = directora.preguntar(pregunta)
    print("Pregunta:", pregunta)
    print()
    print("Respuesta:", resultado.respuesta)
    print()
    print(formatear_registro(resultado.registro))
    print(f"Llamadas: {resultado.llamadas}. Límite alcanzado: {'sí' if resultado.limite_alcanzado else 'no'}.")
    return resultado

### C1: sala del tripulante y distancia

In [ ]:
r1 = preguntar("¿En qué sala está el tripulante y a cuántos pasos está el alienígena?")

### C2: visión del alienígena

In [ ]:
r2 = preguntar("¿El alienígena está viendo al tripulante en este momento?")

### C4: varias herramientas y una acción sobre la partida

In [ ]:
r4 = preguntar("El alienígena lleva rato perdido: dale una pista hacia la sala donde está el tripulante, avanza 10 ticks y dime si se acercó.")

### C5: una sala que no existe

La herramienta devuelve un error con las salas válidas; la directora debe decir que la sala no existe sin inventar una ruta.

In [ ]:
r5 = preguntar("¿Cuántos pasos hay desde la Enfermería hasta el Hangar?")

## Qué pasa al llegar al límite

C6 pide la distancia a las 8 salas, lo que necesita 8 llamadas a `ruta_mas_corta`. Con el límite de 6, el ciclo ejecuta 6, las demás reciben un mensaje de no ejecutada y el modelo responde una última vez sin herramientas, diciendo que alcanzó el límite.

In [ ]:
r6 = preguntar("Calcula la distancia del alienígena a cada una de las 8 salas.", limite=6)

## Cómo leer el registro y dónde está la evidencia

- Cada línea del registro es una herramienta ejecutada: número, nombre con sus argumentos, resultado (recortado si es largo) y tiempo. `ERROR` al final indica que la herramienta devolvió un error, que el modelo recibió como JSON.
- "Llamadas" cuenta herramientas ejecutadas, no turnos del modelo (decisión D9).
- La evidencia completa de los casos (pregunta, valor esperado, llamadas, respuesta, latencia y valoración) está en `experiments/resultados/`, generada con `python -m experiments.casos`. La explicación de cada caso está en `docs/casos.md`.